# Module 17: API & Backend Basics — Solutions

## Complete Solutions for All Exercises

## Part 1: FastAPI Fundamentals — Solutions

In [ ]:
from fastapi import FastAPI
from typing import Optional

# Exercise 1.1: Basic API Setup
app = FastAPI(title='ML Service', version='1.0.0')

@app.get('/')
def root():
    return {'api': 'ML Service', 'status': 'running'}

@app.get('/info')
def info():
    return {'name': 'ML Service', 'version': '1.0.0', 'author': 'ML Engineer'}

@app.get('/items/{item_id}')
def get_item(item_id: int):
    return {'item_id': item_id}

@app.get('/search')
def search(q: Optional[str] = None):
    return {'query': q}

print('Exercise 1.1: All endpoints defined')
print('Endpoints: /, /info, /items/{item_id}, /search')

In [ ]:
# Exercise 1.2: Path and Query Parameters
@app.get('/products/{product_id}')
def get_product(
    product_id: int,
    category: Optional[str] = None,
    page: int = 1,
    limit: int = 10
):
    return {
        'product_id': product_id,
        'category': category,
        'page': page,
        'limit': limit
    }

print('Exercise 1.2: Products endpoint with path + query params')
print('/products/42?category=electronics&page=2&limit=20')

In [ ]:
# Exercise 1.3: Multiple Endpoints (Real Estate Platform)
from fastapi import HTTPException

real_estate_app = FastAPI(title='Real Estate API')

# Mock data
listings = {
    1: {'id': 1, 'address': '123 Main St', 'price': 350000, 'bedrooms': 3, 'bathrooms': 2},
    2: {'id': 2, 'address': '456 Oak Ave', 'price': 520000, 'bedrooms': 4, 'bathrooms': 3},
    3: {'id': 3, 'address': '789 Pine Rd', 'price': 275000, 'bedrooms': 2, 'bathrooms': 1},
}

@real_estate_app.get('/listings')
def get_listings():
    return {'listings': list(listings.values()), 'count': len(listings)}

# IMPORTANT: Static route BEFORE parameterized route
@real_estate_app.get('/listings/search')
def search_listings(
    min_price: Optional[float] = None,
    max_price: Optional[float] = None,
    bedrooms: Optional[int] = None,
    sort_by: Optional[str] = None
):
    results = list(listings.values())
    if min_price:
        results = [h for h in results if h['price'] >= min_price]
    if max_price:
        results = [h for h in results if h['price'] <= max_price]
    if bedrooms:
        results = [h for h in results if h['bedrooms'] >= bedrooms]
    return {'results': results, 'count': len(results)}

@real_estate_app.get('/listings/{listing_id}')
def get_listing(listing_id: int):
    if listing_id not in listings:
        raise HTTPException(status_code=404, detail='Listing not found')
    return listings[listing_id]

print('Exercise 1.3: Real estate endpoints')
print('  GET /listings')
print('  GET /listings/search?min_price=300000&bedrooms=3')
print('  GET /listings/1')

## Part 2: Pydantic Models — Solutions

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import List, Tuple
import uuid

# Exercise 2.1: Input Validation Model
class HouseFeatures(BaseModel):
    bedrooms: int = Field(..., ge=1, le=10)
    bathrooms: float = Field(..., ge=0.5, le=10)
    sqft_living: int = Field(..., ge=100, le=10000)
    sqft_lot: int = Field(..., ge=100, le=100000)
    floors: float = Field(..., ge=1, le=4)
    waterfront: bool = False
    condition: int = Field(..., ge=1, le=5)
    yr_built: int = Field(..., ge=1900, le=2025)

# Test validation
valid_house = HouseFeatures(
    bedrooms=3, bathrooms=2.0, sqft_living=1800, sqft_lot=5000,
    floors=1.5, waterfront=False, condition=3, yr_built=1995
)
print('Exercise 2.1: HouseFeatures model created')
print('Valid house:', valid_house.model_dump())

# Test validation failure
try:
    invalid = HouseFeatures(bedrooms=100, bathrooms=2.0, sqft_living=1800, sqft_lot=5000, floors=1, condition=3, yr_built=1995)
except Exception as e:
    errors = e.errors()
    print('Validation working: caught', errors[0]['msg'] if errors else 'invalid')

In [ ]:
# Exercise 2.2: Response Model
class PricePrediction(BaseModel):
    predicted_price: float
    confidence_interval: Tuple[float, float]
    prediction_id: str
    model_version: str = '1.0.0'
    features_used: List[str]

def predict_house_price(features: HouseFeatures) -> PricePrediction:
    # Mock prediction logic
    base_price = features.sqft_living * 200.0
    price = base_price + (features.bedrooms * 5000) + (features.bathrooms * 10000)
    if features.waterfront:
        price *= 1.5
    
    return PricePrediction(
        predicted_price=round(price, 2),
        confidence_interval=(round(price * 0.9, 2), round(price * 1.1, 2)),
        prediction_id=str(uuid.uuid4()),
        features_used=['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'condition', 'yr_built']
    )

# Test
result = predict_house_price(valid_house)
print('Exercise 2.2: Prediction response')
print(f'  Price: ${result.predicted_price:,.2f}')
print(f'  Confidence: ${result.confidence_interval[0]:,.2f} - ${result.confidence_interval[1]:,.2f}')
print(f'  ID: {result.prediction_id[:8]}...')

In [ ]:
# Exercise 2.3: Custom Validator
class ValidatedHouseFeatures(HouseFeatures):
    @model_validator(mode='after')
    def check_room_count(self):
        if self.bedrooms > 6 and self.sqft_living < 2000:
            raise ValueError(f'Too many bedrooms ({self.bedrooms}) for {self.sqft_living} sqft living area')
        return self

    @field_validator('waterfront')
    @classmethod
    def check_waterfront_age(cls, v, info):
        data = info.data
        if v and 'yr_built' in data and data['yr_built'] < 2000:
            print(f'  [Warning] Waterfront property built in {data["yr_built"]} (may need renovation)')
        return v

# Test valid case
v1 = ValidatedHouseFeatures(
    bedrooms=7, bathrooms=3.0, sqft_living=3500, sqft_lot=10000,
    floors=2.0, waterfront=False, condition=4, yr_built=2010
)
print('Exercise 2.3: Valid case accepted')
print(f'  Bedrooms: {v1.bedrooms}, Sqft: {v1.sqft_living}')
print()

# Test invalid case (too many bedrooms for small house)
try:
    v2 = ValidatedHouseFeatures(
        bedrooms=7, bathrooms=3.0, sqft_living=1500, sqft_lot=10000,
        floors=2.0, waterfront=False, condition=4, yr_built=2010
    )
except Exception as e:
    print('Exercise 2.3: Invalid case rejected')
    print(f'  Error: {e.errors()[0]["msg"]}')

## Part 3: ML Model Serving — Solutions

In [ ]:
import numpy as np
from fastapi import FastAPI, HTTPException

# Exercise 3.1: Model Startup Loading
class MockHouseModel:
    def __init__(self):
        np.random.seed(42)
        self.coefficients = np.random.randn(8) * 50000
        self.feature_names = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'condition', 'yr_built']
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.dot(X, self.coefficients) + 300000

ml_app = FastAPI(title='House Price Predictor')

@ml_app.on_event('startup')
def load_model():
    ml_app.state.model = MockHouseModel()
    ml_app.state.feature_names = MockHouseModel().feature_names
    print('Model loaded at startup')

@ml_app.post('/predict')
def predict(features: HouseFeatures):
    model = ml_app.state.model
    if model is None:
        raise HTTPException(status_code=503, detail='Model not loaded')
    try:
        X = np.array([[features.bedrooms, features.bathrooms, features.sqft_living,
                       features.sqft_lot, features.floors, int(features.waterfront),
                       features.condition, features.yr_built]])
        pred = model.predict(X)[0]
        return {'predicted_price': round(float(pred), 2), 'status': 'success'}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Prediction failed: {str(e)}')

# Simulate startup
load_model()
response = predict(valid_house)
print('Exercise 3.1: ML model serving')
print(f'  Predicted price: ${response["predicted_price"]:,.2f}')

In [ ]:
# Exercise 3.2: Batch Prediction
from pydantic import BaseModel, Field
from typing import List

class BatchHouseFeatures(BaseModel):
    samples: List[HouseFeatures] = Field(..., max_length=32)

class BatchHousePrediction(BaseModel):
    predictions: List[PricePrediction]

@ml_app.post('/predict-batch', response_model=BatchHousePrediction)
def predict_batch(batch: BatchHouseFeatures):
    if len(batch.samples) > 32:
        raise HTTPException(status_code=422, detail='Max batch size is 32')
    model = ml_app.state.model
    if model is None:
        raise HTTPException(status_code=503, detail='Model not loaded')
    results = []
    for s in batch.samples:
        X = np.array([[s.bedrooms, s.bathrooms, s.sqft_living,
                       s.sqft_lot, s.floors, int(s.waterfront),
                       s.condition, s.yr_built]])
        pred = model.predict(X)[0]
        results.append({
            'predicted_price': round(float(pred), 2),
            'features_used': ml_app.state.feature_names
        })
    return {'predictions': results}

# Test batch
batch_req = BatchHouseFeatures(samples=[valid_house, valid_house])
batch_res = predict_batch(batch_req)
print('Exercise 3.2: Batch prediction')
print(f'  Predictions returned: {len(batch_res["predictions"])}')
print(f'  First price: ${batch_res["predictions"][0]["predicted_price"]:,.2f}')

In [ ]:
# Exercise 3.3: Model Versioning

# Create two mock models "v1" and "v2"
class MockModelV1:
    """Simpler model - lower accuracy."""
    def predict(self, X):
        return np.dot(X, np.array([10000, 8000, 150, 5, 5000, 100000, 15000, 300])) + 200000

class MockModelV2:
    """More complex model - higher accuracy."""
    def predict(self, X):
        # Simulates a better model with non-linear terms
        base = np.dot(X, np.array([12000, 10000, 180, 8, 6000, 150000, 20000, 500])) + 250000
        # Add sqft^2 term (non-linear)
        base += X[:, 2] ** 2 * 0.02
        return base

version_app = FastAPI(title='Versioned House Price Predictor')

@version_app.on_event('startup')
def load_models():
    np.random.seed(42)
    version_app.state.models = {
        'v1': {'model': MockModelV1(), 'accuracy': 0.85, 'description': 'Linear model'},
        'v2': {'model': MockModelV2(), 'accuracy': 0.92, 'description': 'Non-linear model with feature interactions'}
    }
    print('Models v1 and v2 loaded')

def get_model(version: str):
    if version not in version_app.state.models:
        raise HTTPException(status_code=404, detail=f'Model version {version} not found')
    return version_app.state.models[version]['model']

@version_app.post('/v1/predict')
def predict_v1(features: HouseFeatures):
    model = get_model('v1')
    X = np.array([[features.bedrooms, features.bathrooms, features.sqft_living,
                   features.sqft_lot, features.floors, int(features.waterfront),
                   features.condition, features.yr_built]])
    pred = model.predict(X)[0]
    return {'version': 'v1', 'predicted_price': round(float(pred), 2)}

@version_app.post('/v2/predict')
def predict_v2(features: HouseFeatures):
    model = get_model('v2')
    X = np.array([[features.bedrooms, features.bathrooms, features.sqft_living,
                   features.sqft_lot, features.floors, int(features.waterfront),
                   features.condition, features.yr_built]])
    pred = model.predict(X)[0]
    return {'version': 'v2', 'predicted_price': round(float(pred), 2)}

@version_app.get('/models')
def list_models():
    return {
        'models': [
            {'version': v, **info}
            for v, info in version_app.state.models.items()
        ]
    }

# Test
load_models()
r1 = predict_v1(valid_house)
r2 = predict_v2(valid_house)
print('Exercise 3.3: Model versioning')
print(f'  {r1["version"]}: ${r1["predicted_price"]:,.2f}')
print(f'  {r2["version"]}: ${r2["predicted_price"]:,.2f}')
print(f'  Models available: {list_models()["models"]}')

## Part 4: Testing — Solutions

In [ ]:
from fastapi.testclient import TestClient

# Create a test app
test_app = FastAPI()
test_app.state.model = MockHouseModel()
test_app.state.feature_names = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'condition', 'yr_built']

@test_app.get('/')
def root():
    return {'api': 'test', 'status': 'running'}

@test_app.get('/health')
def health():
    return {'status': 'healthy'}

@test_app.post('/predict')
def predict(features: HouseFeatures):
    return {'predicted_price': 350000.0, 'prediction_id': 'test-123', 'status': 'success'}

client = TestClient(test_app)

# Exercise 4.1: Test Client
# Test root
response = client.get('/')
assert response.status_code == 200
print('Test 1 passed: GET / returns 200')

# Test health
response = client.get('/health')
assert response.status_code == 200
assert response.json() == {'status': 'healthy'}
print('Test 2 passed: GET /health returns 200 with correct body')

# Test predict with valid data
response = client.post('/predict', json={
    'bedrooms': 3, 'bathrooms': 2.0, 'sqft_living': 1800,
    'sqft_lot': 5000, 'floors': 1.5, 'waterfront': False,
    'condition': 3, 'yr_built': 1995
})
assert response.status_code == 200
assert 'predicted_price' in response.json()
assert 'prediction_id' in response.json()
print('Test 3 passed: POST /predict with valid data returns 200 with expected keys')

# Test predict with invalid data (bedrooms > 10)
response = client.post('/predict', json={
    'bedrooms': 100, 'bathrooms': 2.0, 'sqft_living': 1800,
    'sqft_lot': 5000, 'floors': 1.5, 'waterfront': False,
    'condition': 3, 'yr_built': 1995
})
assert response.status_code == 422
print('Test 4 passed: POST /predict with invalid data returns 422')

# Test missing required fields
response = client.post('/predict', json={'bedrooms': 3})
assert response.status_code == 422
print('Test 5 passed: POST /predict missing fields returns 422')

# Test batch with too many
many_houses = [{
    'bedrooms': 3, 'bathrooms': 2.0, 'sqft_living': 1800,
    'sqft_lot': 5000, 'floors': 1.5, 'waterfront': False,
    'condition': 3, 'yr_built': 1995
} for _ in range(33)]
response = client.post('/predict-batch', json={'samples': many_houses})
assert response.status_code == 422
print('Test 6 passed: Batch with >32 samples returns 422')

print()
print('All tests passed!')

In [ ]:
# Exercise 4.2: Parametrized Tests
# Simulating pytest parametrize manually for notebook environment

test_cases = [
    ('negative bedrooms', {'bedrooms': -1, 'bathrooms': 2.0, 'sqft_living': 1800, 'sqft_lot': 5000, 'floors': 1.5, 'waterfront': False, 'condition': 3, 'yr_built': 1995}),
    ('small sqft', {'bedrooms': 3, 'bathrooms': 2.0, 'sqft_living': 50, 'sqft_lot': 5000, 'floors': 1.5, 'waterfront': False, 'condition': 3, 'yr_built': 1995}),
    ('old year', {'bedrooms': 3, 'bathrooms': 2.0, 'sqft_living': 1800, 'sqft_lot': 5000, 'floors': 1.5, 'waterfront': False, 'condition': 3, 'yr_built': 1800}),
    ('high condition', {'bedrooms': 3, 'bathrooms': 2.0, 'sqft_living': 1800, 'sqft_lot': 5000, 'floors': 1.5, 'waterfront': False, 'condition': 10, 'yr_built': 1995}),
    ('many floors', {'bedrooms': 3, 'bathrooms': 2.0, 'sqft_living': 1800, 'sqft_lot': 5000, 'floors': 5.0, 'waterfront': False, 'condition': 3, 'yr_built': 1995}),
]

print('Exercise 4.2: Parametrized tests')
for name, payload in test_cases:
    response = client.post('/predict', json=payload)
    status = 'PASS' if response.status_code == 422 else 'FAIL'
    print(f'  {status}: {name} -> status {response.status_code}')

print('\nAll parametrized validation tests passed!')

## Part 5: Advanced Topics — Solutions

In [ ]:
# Exercise 5.1: Dependency Injection
from fastapi import Depends, HTTPException
from typing import Dict

# In-memory database dependency
predictions_db: Dict[str, dict] = {}

def get_db():
    return predictions_db

adv_app = FastAPI()

@adv_app.post('/store-prediction')
def store_prediction(prediction: PricePrediction, db=Depends(get_db)):
    db[prediction.prediction_id] = prediction.model_dump()
    return {'message': 'Stored', 'id': prediction.prediction_id}

@adv_app.get('/predictions/{prediction_id}')
def get_prediction(prediction_id: str, db=Depends(get_db)):
    if prediction_id not in db:
        raise HTTPException(status_code=404, detail='Prediction not found')
    return db[prediction_id]

# Test dependencies
test_pred = predict_house_price(valid_house)
store_prediction(test_pred)
retrieved = get_prediction(test_pred.prediction_id)
print('Exercise 5.1: Dependency injection')
print(f'  Stored and retrieved: ${retrieved["predicted_price"]:,.2f}')

In [ ]:
# Exercise 5.2: CORS and Settings
from fastapi.middleware.cors import CORSMiddleware
from pydantic_settings import BaseSettings

class AppSettings(BaseSettings):
    model_path: str = 'model.joblib'
    max_batch_size: int = 32
    enable_logging: bool = True
    allowed_origins: str = 'http://localhost:5173'

    class Config:
        env_file = '.env'

settings = AppSettings()

config_app = FastAPI()

config_app.add_middleware(
    CORSMiddleware,
    allow_origins=settings.allowed_origins.split(','),
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@config_app.get('/settings')
def get_settings():
    # Return sanitized config (no secrets)
    return {
        'model_path': settings.model_path,
        'max_batch_size': settings.max_batch_size,
        'enable_logging': settings.enable_logging,
        'allowed_origins': settings.allowed_origins
    }

# Test
print('Exercise 5.2: CORS and settings configured')
print(f'  CORS origins: {settings.allowed_origins}')
print(f'  Settings: model_path={settings.model_path}, max_batch_size={settings.max_batch_size}')

In [ ]:
# Exercise 5.3: Background Task Logging
from fastapi import BackgroundTasks
from datetime import datetime

prediction_logs = []

def log_prediction_background(prediction_id: str, price: float, bedrooms: int, sqft: int):
    """Background task: log prediction details."""
    entry = {
        'timestamp': datetime.now().isoformat(),
        'prediction_id': prediction_id,
        'price': price,
        'bedrooms': bedrooms,
        'sqft_living': sqft,
        'source_ip': '127.0.0.1'  # Simulated
    }
    prediction_logs.append(entry)
    print(f'  [Background] Logged: ${price:,.2f}')

log_app = FastAPI()

@log_app.post('/predict-with-logging')
def predict_with_logging(features: HouseFeatures, bg_tasks: BackgroundTasks):
    price = features.sqft_living * 200.0 + features.bedrooms * 5000
    pred_id = str(uuid.uuid4())
    
    # Schedule background logging
    bg_tasks.add_task(log_prediction_background, pred_id, price, features.bedrooms, features.sqft_living)
    
    return {
        'predicted_price': round(price, 2),
        'prediction_id': pred_id,
        'status': 'success'
    }

@log_app.get('/logs')
def get_logs():
    return {'logs': prediction_logs[-10:], 'total': len(prediction_logs)}

# Test with manual background task execution
bt = BackgroundTasks()
result = predict_with_logging(valid_house, bt)
print('Exercise 5.3: Background task logging')
print(f'  Prediction returned: ${result["predicted_price"]:,.2f}')
print(f'  Logs count before: {len(prediction_logs)}')
bt()  # Execute background tasks
print(f'  Logs count after: {len(prediction_logs)}')
print(f'  GET /logs returns {len(prediction_logs)} entry/entries')